# London Explorer API Smoke Tests

This notebook tests the documented API endpoints, spatial searches, filters, pagination, validation errors, and the reported boundary request.

## 1) Configure API Base URL and HTTP Helpers

The notebook selects the local backend when available and otherwise uses the configured Render URL.

In [17]:
import json
from typing import Any

import requests

LOCAL_URL = "http://localhost:3000"
RENDER_URL = "https://london-explorer.onrender.com"
TIMEOUT_SECONDS = 30
API_BASE = LOCAL_URL

try:
    health_probe = requests.get(f"{LOCAL_URL}/health", timeout=2)
    if health_probe.status_code != 200:
        raise requests.RequestException
    print(f"Using local API: {API_BASE}")
except requests.RequestException:
    API_BASE = RENDER_URL
    print(f"Using Render API: {API_BASE}")

results: list[dict[str, Any]] = []


def record(name: str, passed: bool, detail: str = "") -> None:
    results.append({"name": name, "passed": passed, "detail": detail})


def get_response(path: str, params: dict[str, Any] | None = None) -> requests.Response:
    return requests.get(f"{API_BASE}{path}", params=params, timeout=TIMEOUT_SECONDS)


def call_api(path: str, params: dict[str, Any] | None = None) -> Any:
    response = get_response(path, params)
    response.raise_for_status()
    return response.json()


def expect_status(name: str, path: str, params: dict[str, Any], expected: int) -> requests.Response:
    response = get_response(path, params)
    passed = response.status_code == expected
    detail = f"HTTP {response.status_code}; {response.text[:300]}"
    record(name, passed, detail)
    if not passed:
        print(f"FAIL: {name}: {detail}")
    return response


def preview(payload: Any, max_items: int = 3) -> None:
    if isinstance(payload, dict) and isinstance(payload.get("data"), list):
        print(json.dumps({key: value for key, value in payload.items() if key != "data"}, indent=2))
        print(json.dumps(payload["data"][:max_items], indent=2))
    else:
        print(json.dumps(payload, indent=2)[:2000])


def geometry_params(search_type: str, geometry: dict[str, Any], radius_m: int | None = None) -> dict[str, Any]:
    params: dict[str, Any] = {
        "search_type": search_type,
        "geometry": json.dumps(geometry, separators=(",", ":")),
        "venue_type": "",
        "score_basis": 0,
        "score_tier": 0,
    }
    if radius_m is not None:
        params["radius_m"] = radius_m
    return params

Using local API: http://localhost:3000


## 2) Test Health Endpoint

Call `/health`, require HTTP 200, and verify the response is JSON.

In [18]:
health_response = expect_status("health", "/health", {}, 200)
health_payload = health_response.json()
assert isinstance(health_payload, dict)
print(f"Health response: {health_payload}")

Health response: {'status': 'ok'}


## 3) Test Places Top Endpoint

Test the default top-places response and validate its collection shape.

In [19]:
top_response = expect_status(
    "places top default viewport",
    "/api/places/top",
    {"sw_lat": 51.49, "sw_lng": -0.16, "ne_lat": 51.53, "ne_lng": -0.09, "limit": 10},
    200,
)
top_payload = top_response.json()
assert isinstance(top_payload.get("data"), list)
assert top_payload.get("limit") == 10
print("Top places preview:")
preview(top_payload)

Top places preview:
{
  "total": 10,
  "limit": 10
}
[
  {
    "id": "ChIJ5x5n-d0bdkgR5wLl0cOlPEM",
    "restaurant_name": "Tofu Vegan Charlotte Street",
    "cuisine_type": "Vegetarian & Vegan",
    "lat": 51.5190574,
    "lon": -0.1350817,
    "normal_1": 0.9999451637268066,
    "rank": 4
  },
  {
    "id": "ChIJWT8faDgbdkgRvAuaJPO8udM",
    "restaurant_name": "Bench Bistro",
    "cuisine_type": "Cafe & Coffee",
    "lat": 51.5276464,
    "lon": -0.1189174999999999,
    "normal_1": 0.9998354911804199,
    "rank": 4
  },
  {
    "id": "ChIJGXcxWloFdkgRTvWv6q6p4z4",
    "restaurant_name": "Falafel Zaki Zaki",
    "cuisine_type": "Middle Eastern",
    "lat": 51.4974087,
    "lon": -0.134047,
    "normal_1": 0.9997806549072266,
    "rank": 4
  }
]


## 4) Test Places List Endpoint

Test pagination metadata and restaurant records.

In [20]:
list_response = expect_status(
    "places list page 1 viewport",
    "/api/places/list",
    {"sw_lat": 51.49, "sw_lng": -0.16, "ne_lat": 51.53, "ne_lng": -0.09, "page": 1, "page_size": 10},
    200,
)
list_payload = list_response.json()
assert list_payload.get("page") == 1
assert isinstance(list_payload.get("data"), list)
print("Restaurant list preview:")
preview(list_payload)

Restaurant list preview:
{
  "page": 1,
  "page_size": 10
}
[
  {
    "id": "ChIJ5x5n-d0bdkgR5wLl0cOlPEM",
    "lat": 51.5190574,
    "lon": -0.1350817,
    "ranking": 0.9999451637268066,
    "display_name": "Tofu Vegan Charlotte Street",
    "cuisine_type": "Vegetarian & Vegan",
    "price": "20+",
    "distance_m": null,
    "is_chain": false,
    "is_major_chain": false,
    "venue_type": "Dine-In",
    "google_maps_uri": "https://maps.google.com/?cid=4844929559602463463&g_mp=Cilnb29nbGUubWFwcy5wbGFjZXMudjEuUGxhY2VzLlNlYXJjaE5lYXJieRACGAQgAA",
    "website_uri": "https://www.tofuvegan.com/"
  },
  {
    "id": "ChIJWT8faDgbdkgRvAuaJPO8udM",
    "lat": 51.5276464,
    "lon": -0.1189174999999999,
    "ranking": 0.9998354911804199,
    "display_name": "Bench Bistro",
    "cuisine_type": "Cafe & Coffee",
    "price": "10+",
    "distance_m": null,
    "is_chain": false,
    "is_major_chain": false,
    "venue_type": "Dine-In",
    "google_maps_uri": "https://maps.google.com/?cid=15256432

## 5) Test Boundary Spatial Search

Send Polygon and MultiPolygon GeoJSON as encoded query parameters to both place endpoints.

In [21]:
polygon = {
    "type": "Polygon",
    "coordinates": [[
        [-0.15, 51.50], [-0.10, 51.50], [-0.10, 51.52],
        [-0.15, 51.52], [-0.15, 51.50],
    ]],
}

multipolygon = {
    "type": "MultiPolygon",
    "coordinates": [[[[-0.15, 51.50], [-0.10, 51.50], [-0.10, 51.52], [-0.15, 51.52], [-0.15, 51.50]]]],
}

boundary_params = geometry_params("boundary", polygon)
for path, name, extra in [
    ("/api/places/top", "boundary top polygon", {"limit": 10}),
    ("/api/places/list", "boundary list polygon", {"page": 1, "page_size": 10}),
]:
    response = expect_status(name, path, {**boundary_params, **extra}, 200)
    assert isinstance(response.json().get("data"), list)

multipolygon_response = expect_status(
    "boundary top multipolygon",
    "/api/places/top",
    {**geometry_params("boundary", multipolygon), "limit": 10},
    200,
)
assert isinstance(multipolygon_response.json().get("data"), list)
print("Boundary Polygon and MultiPolygon checks passed.")

Boundary Polygon and MultiPolygon checks passed.


## 6) Test Street Spatial Search

Send LineString and MultiLineString geometries with a positive `radius_m`.

In [22]:
line = {
    "type": "LineString",
    "coordinates": [[-0.142, 51.501], [-0.130, 51.505], [-0.118, 51.510]],
}
multiline = {
    "type": "MultiLineString",
    "coordinates": [[[-0.142, 51.501], [-0.130, 51.505]], [[-0.130, 51.505], [-0.118, 51.510]]],
}

street_params = geometry_params("street", line, radius_m=250)
for path, name, extra in [
    ("/api/places/top", "street top linestring", {"limit": 10}),
    ("/api/places/list", "street list linestring", {"page": 1, "page_size": 10}),
]:
    response = expect_status(name, path, {**street_params, **extra}, 200)
    assert isinstance(response.json().get("data"), list)

multiline_response = expect_status(
    "street top multilinestring",
    "/api/places/top",
    {**geometry_params("street", multiline, radius_m=250), "limit": 10},
    200,
)
assert isinstance(multiline_response.json().get("data"), list)
print("Street LineString and MultiLineString checks passed.")

Street LineString and MultiLineString checks passed.


## 7) Test Geometry Search Across Count and Histograms

Exercise boundary and street search masks on the count, cuisine histogram, and cost histogram APIs. Heatmap remains a viewport-bbox API and is tested separately.

In [23]:
geometry_api_cases = [
    ("boundary places count", "/api/places/count", {"scope": "citywide", **geometry_params("boundary", polygon)}, "count", int),
    ("boundary cuisine histogram", "/api/cuisine_histogram", {"scope": "citywide", **geometry_params("boundary", polygon)}, "cuisine_histogram", list),
    ("boundary cost histogram", "/api/cost_histogram", {"scope": "citywide", **geometry_params("boundary", polygon)}, "cost_histogram", list),
    ("street places count", "/api/places/count", {"scope": "citywide", **geometry_params("street", line, radius_m=250)}, "count", int),
    ("street cuisine histogram", "/api/cuisine_histogram", {"scope": "citywide", **geometry_params("street", line, radius_m=250)}, "cuisine_histogram", list),
    ("street cost histogram", "/api/cost_histogram", {"scope": "citywide", **geometry_params("street", line, radius_m=250)}, "cost_histogram", list),
]

for name, path, params, key, expected_type in geometry_api_cases:
    response = expect_status(name, path, params, 200)
    assert isinstance(response.json().get(key), expected_type)

heatmap_response = expect_status(
    "heatmap viewport bbox",
    "/api/heatmap",
    {"sw_lat": 51.49, "sw_lng": -0.16, "ne_lat": 51.53, "ne_lng": -0.09},
    200,
)
assert isinstance(heatmap_response.json().get("data"), list)
print("Geometry count/histogram and viewport heatmap checks passed.")

Geometry count/histogram and viewport heatmap checks passed.


## 8) Test Query Filters and Pagination

Exercise cuisine, cost, venue, score, limit, page, and page-size parameters across active endpoints.

In [24]:
filtered_params = {
    "cuisine": "Italian",
    "cost": "20+",
    "venue_type": "Dine-In",
    "score_basis": 2,
    "score_tier": 2,
}

filtered_top = expect_status(
    "filtered top places",
    "/api/places/top",
    {**filtered_params, "sw_lat": 51.49, "sw_lng": -0.16, "ne_lat": 51.53, "ne_lng": -0.09, "limit": 5},
    200,
)
assert filtered_top.json().get("limit") == 5

filtered_list = expect_status(
    "filtered list page 2",
    "/api/places/list",
    {**filtered_params, "sw_lat": 51.49, "sw_lng": -0.16, "ne_lat": 51.53, "ne_lng": -0.09, "page": 2, "page_size": 5},
    200,
)
assert filtered_list.json().get("page") == 2
assert filtered_list.json().get("page_size") <= 5

count_response = expect_status(
    "filtered places count",
    "/api/places/count",
    {"scope": "citywide", **filtered_params, "requestTierRep": "true"},
    200,
)
assert isinstance(count_response.json().get("count"), int)

for path, name, extra, key in [
    ("/api/cuisine_histogram", "filtered cuisine histogram", {"scope": "citywide", "cost": "20+", "venue_type": "Dine-In", "score_basis": 2, "score_tier": 2}, "cuisine_histogram"),
    ("/api/cost_histogram", "filtered cost histogram", {"scope": "citywide", "cuisine": "Italian", "venue_type": "Dine-In", "score_basis": 2, "score_tier": 2}, "cost_histogram"),
]:
    response = expect_status(name, path, extra, 200)
    assert isinstance(response.json().get(key), list)

print("Filter and pagination checks passed.")

Filter and pagination checks passed.


## 9) Test Validation and Error Responses

Verify malformed geometry, unsupported geometry/search types, missing street radius, and invalid numeric parameters return HTTP 422.

In [25]:
validation_cases = [
    ("unsupported boundary geometry", "/api/places/top", {**geometry_params("boundary", line), "limit": 10}),
    ("malformed GeoJSON", "/api/places/top", {"search_type": "boundary", "geometry": "not-json", "limit": 10}),
    ("missing street radius", "/api/places/top", geometry_params("street", line)),
    ("invalid search type", "/api/places/top", {"search_type": "circle", "geometry": json.dumps(polygon), "limit": 10}),
    ("invalid top limit", "/api/places/top", {"limit": 0}),
    ("invalid list page", "/api/places/list", {"sw_lat": 51.49, "sw_lng": -0.16, "ne_lat": 51.53, "ne_lng": -0.09, "page": 0}),
    ("invalid list page size", "/api/places/list", {"sw_lat": 51.49, "sw_lng": -0.16, "ne_lat": 51.53, "ne_lng": -0.09, "page_size": 0}),
    ("invalid score basis", "/api/places/count", {"scope": "citywide", "score_basis": 4}),
    ("invalid score tier", "/api/cost_histogram", {"scope": "citywide", "score_tier": 5}),
]

for name, path, params in validation_cases:
    response = expect_status(name, path, params, 422)
    print(f"{name}: {response.text[:300]}")

print("Validation checks passed.")

unsupported boundary geometry: {"detail":"boundary geometry must be one of: MultiPolygon, Polygon"}
malformed GeoJSON: {"detail":"geometry must be valid GeoJSON"}
missing street radius: {"detail":"radius_m must be greater than zero for a street search"}
invalid search type: {"detail":"search_type must be 'boundary' or 'street'"}
invalid top limit: {"detail":[{"type":"greater_than_equal","loc":["query","limit"],"msg":"Input should be greater than or equal to 1","input":"0","ctx":{"ge":1}}]}
invalid list page: {"detail":[{"type":"greater_than_equal","loc":["query","page"],"msg":"Input should be greater than or equal to 1","input":"0","ctx":{"ge":1}}]}
invalid list page size: {"detail":[{"type":"greater_than_equal","loc":["query","page_size"],"msg":"Input should be greater than or equal to 1","input":"0","ctx":{"ge":1}}]}
invalid score basis: {"detail":[{"type":"less_than_equal","loc":["query","score_basis"],"msg":"Input should be less than or equal to 2","input":"4","ctx":{"le":2}}]}
inv

## 10) Test Reported Boundary Request

Reproduce the reported Polygon request with `score_basis=2`, `score_tier=2`, `limit=20`, and no empty cuisine or cost filters.

In [26]:
reported_boundary = {
    "type": "Polygon",
    "coordinates": [[
        [-0.13297855854034424, 51.51164147802153],
        [-0.13143930584192276, 51.51228748783224],
        [-0.12993291020393372, 51.51285023503663],
        [-0.12937869876623154, 51.51317260655467],
        [-0.12944072484970093, 51.51318721232994],
        [-0.12951113283634186, 51.513347666895726],
        [-0.12942194938659668, 51.5135967980924],
        [-0.13015251606702805, 51.51526765284837],
        [-0.1306886225938797, 51.51513850180044],
        [-0.13088978826999664, 51.51548339061567],
        [-0.1307959109544754, 51.51550488087889],
        [-0.13095080852508545, 51.51559355418786],
        [-0.13128407299518585, 51.5160913733716],
        [-0.13126764446496964, 51.51612788546136],
        [-0.13070136308670044, 51.51618338378175],
        [-0.13073086738586426, 51.51636510901172],
        [-0.13295140117406845, 51.51621301067698],
        [-0.14014039188623428, 51.515455641037505],
        [-0.14176614582538605, 51.51522821904541],
        [-0.14182951301336288, 51.51509656414393],
        [-0.14162197709083557, 51.51449482754723],
        [-0.14074087142944336, 51.51331657756367],
        [-0.13813477009534836, 51.51044312225777],
        [-0.13766169548034668, 51.5101850011058],
        [-0.1369609683752067, 51.51000199920914],
        [-0.1363283023238182, 51.50996631688366],
        [-0.13526279479265213, 51.51012240076776],
        [-0.13481922447681427, 51.510284952800575],
        [-0.1345553621649742, 51.5102188052526],
        [-0.13440247625112534, 51.51023445531203],
        [-0.13381808996200562, 51.51096499410466],
        [-0.13348516076803207, 51.511293012853784],
        [-0.13297855854034424, 51.51164147802153],
    ]],
}

reported_params = geometry_params("boundary", reported_boundary)
reported_params.update({"score_basis": 2, "score_tier": 2})

reported_top = expect_status(
    "reported boundary top",
    "/api/places/top",
    {**reported_params, "limit": 20},
    200,
)
reported_list = expect_status(
    "reported boundary list",
    "/api/places/list",
    {**reported_params, "page": 1, "page_size": 20},
    200,
)
assert isinstance(reported_top.json().get("data"), list)
assert isinstance(reported_list.json().get("data"), list)
print("Reported boundary request passed for both place endpoints.")

Reported boundary request passed for both place endpoints.


## 11) Run Endpoint Smoke-Test Summary

Summarize every recorded check and fail the notebook if any required test failed.

In [27]:
passed = [result for result in results if result["passed"]]
failed = [result for result in results if not result["passed"]]

print(f"API base: {API_BASE}")
print(f"Passed: {len(passed)}")
print(f"Failed: {len(failed)}")
for result in results:
    marker = "PASS" if result["passed"] else "FAIL"
    print(f"{marker}: {result['name']}")
    if not result["passed"]:
        print(f"  {result['detail']}")

assert not failed, f"{len(failed)} API smoke tests failed"
print("All API smoke tests passed.")

API base: http://localhost:3000
Passed: 32
Failed: 0
PASS: health
PASS: places top default viewport
PASS: places list page 1 viewport
PASS: boundary top polygon
PASS: boundary list polygon
PASS: boundary top multipolygon
PASS: street top linestring
PASS: street list linestring
PASS: street top multilinestring
PASS: boundary places count
PASS: boundary cuisine histogram
PASS: boundary cost histogram
PASS: street places count
PASS: street cuisine histogram
PASS: street cost histogram
PASS: heatmap viewport bbox
PASS: filtered top places
PASS: filtered list page 2
PASS: filtered places count
PASS: filtered cuisine histogram
PASS: filtered cost histogram
PASS: unsupported boundary geometry
PASS: malformed GeoJSON
PASS: missing street radius
PASS: invalid search type
PASS: invalid top limit
PASS: invalid list page
PASS: invalid list page size
PASS: invalid score basis
PASS: invalid score tier
PASS: reported boundary top
PASS: reported boundary list
All API smoke tests passed.
